In [5]:
import cv2
import numpy as np

In [27]:
'''
Añadir restar fondo
Añadir fluctuaciones para contar con la inestabilidad del laser
Empaquetar dataset con h5py

'''


def prepocessing(image_path, roi_width=128):
    '''
    '''
    
    # Cargar la imagen en escala de grises (1 canal)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE | cv2.IMREAD_ANYDEPTH)
    if img is None:
        raise ValueError("No se pudo cargar la imagen. Verifica la ruta.")
    
    img_float = img.astype(np.float32)
    #img_limpia = img_float - fondo_float
    #img_limpia = np.clip(img_limpia, 0, None)
    img_height, ancho_img = img.shape


    # 1. ROI Center Search
    # Fusing peaks and finding CM of pattern with Gaussian filter
    img_blur = cv2.GaussianBlur(img, (51, 51), 0)
    _, _, _, max_loc = cv2.minMaxLoc(img_blur)
    cx, cy = max_loc

    # 2. Recorte de la Región de Interés (ROI)
    mitad_ancho = roi_width // 2
    
    # Calcular límites asegurando que no excedan las dimensiones de la imagen original
    x1 = max(0, cx - mitad_ancho)
    x2 = min(ancho_img, cx + mitad_ancho)
    
    roi = img_float[:, x1:x2]
    
    # Si el patrón está muy cerca del borde, rellenar el área faltante con ceros para mantener el tamaño fijo
    roi_rectangular = np.zeros((img_height, roi_width), dtype=np.float32)
    roi_rectangular[:, 0:(x2-x1)] = roi

    #Vertical alineation
    centro_objetivo = img_height // 2
    desplazamiento_y = centro_objetivo -cy
    
    # Affine translation: no X movement and Y displacement. New borders are 0
    M = np.float([
        [1, 0, 0],
        [0, 1, desplazamiento_y]
    ])
    roi_alineada = cv2.warpAffine(roi_rectangular, M, (roi_width, img_height))

    # 3. Dynamic range compression (Log scale)
    # log1p <---> log(1 + x) so no log(0) on black background
    roi_log = np.log1p(roi_rectangular)

    # 4. Min-Max [0, 1] range normalization
    val_min = np.min(roi_log)
    val_max = np.max(roi_log)
    
    if val_max > val_min:
        roi_norm = (roi_log - val_min) / (val_max - val_min)
    else:
        roi_norm = roi_log  # Previene división por cero en imágenes completamente negras

    return roi_norm

# === Ejemplo de ejecución ===
if __name__ == "__main__":
    ruta_entrada = "imagenes_microscopio/gs_70MHz_bdwth_centered_91MHz_rand1_snap_HamaCAM.tiff"
    
    # Obtener el tensor 2D procesado (listo para PyTorch/TensorFlow)
    tensor_entrenamiento = prepocessing(ruta_entrada, roi_width=128)
    
    # Para visualizar el resultado, se re-escala a 8 bits (0-255)
    imagen_visual = (tensor_entrenamiento * 255).astype(np.uint8)
    cv2.imwrite("roi_preprocesada.jpg", imagen_visual)
    print("Imagen preprocesada guardada con éxito.")

Imagen preprocesada guardada con éxito.


In [28]:
print(imagen_visual.shape)

(2304, 128)


In [15]:
ruta_entrada = "/Documents/Universidad/TFG/data/example/imagenes_microscopio/gs_70MHz_bdwth_centered_91MHz_rand1_snap_HamaCAM.tiff"


In [22]:
import os
carpeta = os.getcwd()

print(carpeta)

/home/guillermo/Documents/Universidad/TFG/data/example
